# P(d) long-range run — subtitles corpora (Llama-3.1-8B)

Runs the main marginal persistence pipeline (ordered vs whole-prefix-shuffle) on
the subtitle-dialogue corpora so they gain an alpha(P) for the range-matched
beta-vs-alpha comparison with Exp 2. Mirrors Corpus_Expansion_LongRange_Llama;
writes to Results/corpus_expansion_longrange/llama, where the Exp 2 analysis
reads alpha. Cached corpora skip.

In [ ]:
!pip install -q -U accelerate

In [ ]:
import numpy as np
import json, math, random, time
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
BASE = DRIVE / 'Results/corpus_expansion_longrange/llama'   # same dir alpha_from_main reads
BASE.mkdir(parents=True, exist_ok=True)
TARGETS_PATH = DRIVE / 'Results/corpus_expansion/targets_llama.jsonl'
TOK_MANIFEST_PATH = DRIVE / 'Results/corpus_expansion/tokenized_manifest_llama.jsonl'

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
CTX_LENGTHS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
MAX_CTX = max(CTX_LENGTHS); TARGET_LEN = 30; TARGET_FRACS = [0.5]
MIN_DOC_TOK = MAX_CTX + TARGET_LEN + 50; N_SHUFFLES = 1; SEED = 20260503

RUN_CORPORA = ['subtitles_dialogue_en','subtitles_dialogue_de','subtitles_dialogue_fr',
               'subtitles_dialogue_ru','subtitles_dialogue_tr','subtitles_dialogue_es',
               'subtitles_dialogue_ar']
print('P(d) run for:', RUN_CORPORA)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
model.eval(); print('loaded')

In [ ]:
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2: return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks); ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1); nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0: return float('inf'), float('inf')
    mn = nll / cnt; return math.exp(mn), mn

def compute_longrange_curves(full_ids, ts, te):
    tgt = full_ids[ts:te]; o_ppl, s_ppl = [], []
    for c in CTX_LENGTHS:
        pfx = [] if c == 0 else full_ids[ts - c:ts]
        p_ord, _ = ppl_nll(pfx, tgt); o_ppl.append(p_ord)
        if c == 0: s_ppl.append(p_ord)
        else:
            rng = random.Random(SEED + c); sh_ppls = []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                p_sh, _ = ppl_nll(sh, tgt)
                if not math.isinf(p_sh): sh_ppls.append(p_sh)
            s_ppl.append(np.mean(sh_ppls) if sh_ppls else p_ord)
    return {'context_lengths': list(CTX_LENGTHS), 'ordered_ppl': o_ppl, 'shuffled_ppl': s_ppl}
print('pipeline ready')

In [ ]:
# Resolver (corpus_expansion loader; subtitles are ce corpora).
tok_manifest = {}
if TOK_MANIFEST_PATH.exists():
    for line in open(TOK_MANIFEST_PATH):
        d = json.loads(line); tok_manifest[d['document_id']] = d['file_path']
ce_corpora = {}
for line in open(TARGETS_PATH):
    t = json.loads(line); ce_corpora.setdefault(t['corpus_id'], set()).add(t['document_id'])
def fix_ce_path(p): return str(p).replace('data/corpus_expansion/clean/', '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')
def get_doc_texts(cid):
    for doc_id in ce_corpora.get(cid, set()):
        fp = tok_manifest.get(doc_id)
        if fp is None: continue
        ap = Path(fix_ce_path(fp))
        if not ap.exists(): ap = Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion') / Path(fp).name
        try: yield doc_id, ap.read_text(encoding='utf-8', errors='replace').strip()
        except Exception: continue
for c in RUN_CORPORA: print(f'  {c}: {sum(1 for _ in get_doc_texts(c))} docs')

In [ ]:
for corpus_id in RUN_CORPORA:
    cache = BASE / f'{corpus_id}.json'
    if cache.exists(): print(f'{corpus_id}: cached ({len(json.load(open(cache)))})'); continue
    docs = list(get_doc_texts(corpus_id))
    if not docs: print(f'{corpus_id}: no docs'); continue
    print(f'\n{"="*60}\n{corpus_id} ({len(docs)} docs)\n{"="*60}')
    t0 = time.time(); results = []; skipped = 0
    for doc_id, text in tqdm(docs, desc=corpus_id):
        full_ids = tokenizer.encode(text, add_special_tokens=False); n = len(full_ids)
        if n < MIN_DOC_TOK: skipped += 1; continue
        rem_start, rem_end = MAX_CTX, n - TARGET_LEN
        for frac in TARGET_FRACS:
            ts = int(rem_start + frac * (rem_end - rem_start)); te = ts + TARGET_LEN
            if ts - MAX_CTX < 0 or te > n: continue
            r = compute_longrange_curves(full_ids, ts, te)
            r['corpus_id'] = corpus_id; r['document_id'] = doc_id
            r['target_id'] = f'{doc_id}__pos{int(frac*100):02d}'; results.append(r)
    json.dump(results, open(cache, 'w'))
    print(f'  {len(results)} targets in {(time.time()-t0)/60:.1f} min ({skipped} short-skipped)')
print('\nDone. Re-run Exp2_Qd_Analysis to get matched alpha for the subtitles cells.')